<a href="https://colab.research.google.com/github/ShahidMazumdarAI-hub/FINE_TUNING_LLM/blob/main/Fine_tuning_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

In [ ]:
#now we will define some parameters
max_seq_length = 2048
dtype = None
load_in_4bit = True

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(   #here at this point we are getting a pre-trained model, here we are fine tuning the Llama 3.2 model
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=max_seq_length, #max_seq_length is maximum seq length(in tokens) means how much longer sequence the model can process, here the model can process sequences upto 2048 tokens
    dtype=dtype,  #in the above dtype=None means let the GPU figure out what kind of dtype will be the best, this is basically the datatype for weights and computations. Automatically selects the appropriate data type based on the hardware.. Like torch.float16 uses 16-bit floating point precision, reducing memory usage and increasing speed on compatible GPUs
    load_in_4bit=load_in_4bit #determines whether to load the model using 4-bit quantization , ideal for situations where memory efficiency is crucial

    )

In [ ]:
#now i have my Llama model here in this above model parameter and i will fine-tune this model and i will be also needing TOKENIZER
model = FastLanguageModel.get_peft_model(  #here i get the PARAMETER FINE-TUNED(PEFT) Model, in here i m defining the parameters for LoRA like see below
    model, #i loaded my L lama 3 model. in the below i defined all the parameters for LoRA
    r = 16,  #rank is my hyper-parameter like it also can be 8, 16, 32, 64, 128. Hyper-parameter is instrumental for fast efficiency as because it updates lesser number of parameters than the actual set of matrices which fetches much more parameters
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj" ,   #in the transformer architecture ,the attention layer has 4 matrices (Wq, Wk, Wv, Wo) and in FeedForward layer we have 2 matrices(W1, W2), thus here q_proj is for Wq, k_proj is for Wk etc etc. If u dont specify these parameters then the extra delta W will not be added where this extra Delta W's parameters will be updated keeping the W frozen
                      "gate_proj", "up_proj", "down_proj" ,],
    lora_alpha = 16,  #this Lora_alpha is something from the mathematical equation formula of Quantization, this will tell you that how much the extra delta W contributes to your weight updates, means how much delta W contributes to the final weights
    lora_dropout = 0,  #dropout is for regularization. Dropout is a regularization technique that randomly sets a fraction of input units to zero at each update during training. Here lora_dropout = 0 means that no dropout is being applied during the LoRA fine-tuning.
    bias = "none",  #during the LoRA fine-tuning process, the bias terms of the linear layers in the model(W) will not be updated. Only the LoRA weights (the low-rank matrices A and B or delta W which when decomposed or broken down into A and B with hyper-parameter r) are trained, while the original bias terms remain frozen. This further reduces the number of trainable parameters, making the fine-tuning even more efficient.
    use_gradient_checkpointing = "unsloth", #is a crucial parameter for memory efficiency during training. Gradient checkpointing is a technique that reduces memory consumption by trading computation for memory. Instead of storing all intermediate activations for backpropagation, it recomputes them during the backward pass. When set to "unsloth", it indicates that Unsloth's optimized version of gradient checkpointing is being used, which is designed to further enhance memory savings and potentially speed up training for large models, especially when fine-tuning with LoRA.
    random_state = 3407,  # is used to ensure reproducibility.  if you run the same code with the same random_state value multiple times, you will get the exact same results, which is crucial for debugging, comparing different models, and ensuring experiments are repeatable.
    use_rslora = False, #indicates that the model is not utilizing RSLora. RSLora, or Rank-Stabilized LoRA, is an extension of the LoRA technique designed to improve stability and performance
    loftq_config = None, #means that the LoftQ (Low-Rank Quantization) technique is not being used. LoftQ is a method that combines quantization with LoRA to further reduce the memory footprint of fine-tuning large language models. It typically involves quantizing the pretrained model weights to a lower bit-width and then applying LoRA.
)

In [ ]:
#Now i will download the DATASET on which i wanna fine-tune my above Llama model, here we will use ServiceNow-AI/R1-Distill-SFT dataset
from datasets import load_dataset  #datasets is a hugging face library
dataset = load_dataset("ServiceNow-AI/R1-Distill-SFT", 'v0', split="train")

#ServiceNow-AI/R1-Distill-SFT is a dataset in huggingface which has 172K rows where each row has a problem with solutions, cumulatively multiple problems with solutions
#Thus we are fine-tuning our Llama model on this data which is having solutions to multiple various breed of problems all stored in 172k rows
#thus our model can do better reasoning now


In [ ]:
#i have loaded my dataset, let's see some of the rows(problems+solutions) of ServiceNow-AI/R1-Distill-SFT dataset
dataset[:2]

In [ ]:
#now we will take all the 172K records of our dataset which will be used for fine-tuning the LLM(Llama)
#here we will take each of those records(rows) and put in the prompt below
#below <problem> of r1_prompt there are 3 curly braces
r1_prompt = """You are reflective helpful assistant engaged in thorough iterative reasoning , mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
{}
</problem>

{}
{}
"""
EOS_TOKEN = tokenizer.eos_token #EOS_TOKEN stands for 'End-Of-Sequence Token'. It's a special token provided by the tokenizer (which in this case is likely a tokenizer for the Llama 3.2 model). Language models like Llama use EOS_TOKEN to signal the end of a generated text sequence.

def formatting_prompts_func(examples):
  problems = examples["problem"]  #line1
  thoughts = examples["reannotated_assistant_content"] #line2
  solutions = examples["solution"] #line3
  texts = []

  for problem, thought, solution in zip(problems, thoughts, solutions):
    text = r1_prompt.format(problem, thought, solution) + EOS_TOKEN  #when we do .format then in first curly brace above, it will put problem(line1), in second curly brace it will put thought(line2) and in third curly brace it will put solution(line3) and finally it will return you the text(line4)
    texts.append(text)

  return {"text": texts}  #line4

dataset = dataset.map(formatting_prompts_func, batched = True,) #.map func will go through each record or each row of our ServicesAI dataset one by one and for each record the func def formatting_prompts_func is called and in every record or row we have problem, solution and deep-seek like step by step thought process
                                                                #here batched = True is for performance optimization. It means that the formatting_prompts_func function will be applied to multiple examples (a batch) from the dataset ServicesAI at once, rather than processing each example individually.


In [ ]:
#now we are creating the trainer object, trainer obj is coming from transformer reinforcement learning library which is from huggingFace
#in this library we have the SFTTrainer(supervised Fine-Tuned training trainer) which we are importing from transformer reinforcement learning library(trl)
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model, #specifies the LoRA-adapted Llama model that you previously configured.
    tokenizer=tokenizer, #The tokenizer associated with your Llama model is passed, which is crucial for processing text into a format the model understands.
    train_dataset=dataset, #This is the ServiceNow-AI/R1-Distill-SFT dataset that you prepared and formatted earlier, which will be used to train the model.
    dataset_text_field="text", #This tells the trainer that the column containing the formatted prompts (problem, thought, and solution) in your dataset is named "text".
    max_seq_length=max_seq_length, #This parameter ensures that all input sequences are truncated or padded to the maximum sequence length (2048 in your case), which was defined earlier.
    dataset_num_proc=2, #This means that 2 processes will be used to preprocess the dataset, potentially speeding up data loading.
    packing=False, #When packing=False, the trainer does not combine multiple short examples into a single longer sequence to fill max_seq_length. This ensures each example is processed individually.
    args = TrainingArguments(              #all the training hyperparameters are defined here
        per_device_train_batch_size=2,     #Each GPU will process 2 examples at a time.
        gradient_accumulation_steps=4,     # Gradients are accumulated over 4 steps before a weight update is performed. This effectively creates a larger batch size for gradient calculation (2 * 4 = 8).
        warmup_steps=5,                    # The learning rate will gradually increase for the first 5 steps before reaching its full value.
        max_steps=60,                      # The training will run for a total of 60 optimization steps.
        learning_rate=2e-4,                # The initial learning rate for the optimizer.
        fp16=not is_bfloat16_supported(),  # This enables mixed-precision training, using either 16-bit floating point (fp16) or BFloat16 (bf16) depending on your GPU's capabilities. This can speed up training and reduce memory usage.
        bf16=is_bfloat16_supported(),
        logging_steps=1,                   # Training progress (like loss) will be logged every step.
        optim="adamw_8bit",                # The AdamW optimizer is used with 8-bit quantization, which is memory-efficient.
        weight_decay=0.01,                 # A regularization technique to prevent overfitting by penalizing large weights.
        lr_scheduler_type="linear",        # The learning rate will linearly decay after the warmup phase.
        seed=3407,                         # A fixed random seed for reproducibility of the training process.
        output_dir="outputs",              # The directory where the training artifacts (like checkpoints) will be saved.
        report_to="none"                   # This disables reporting of training metrics to external services like Weights & Biases.
    ),
)

In [ ]:
trainer_stats = trainer.train()  #in the o/p we will see that the loss gets reduced eventually which is a good thing which means my model is training in a proper direction

In [ ]:
#Our model is finally trained
#now let's try a prompt
from unsloth.chat_templates import get_chat_template
sys_prompt =  """You are reflective helpful assistant engaged in thorough iterative reasoning , mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer.
<problem>
{}
</problem>

{}
{}
"""

message = sys_prompt.format("If there are 8 people in a room with all male members. Then how many female members are present in the room?", "", "")  #This mi8 look like a normal problem but these are harder for LLMs, since we fine-tuned our LLM on the dataset, thus now our LLM can process thoughts like DEEP SEEK. The remaining two "" "" This fixes the IndexError by providing two additional empty strings for the thought and solution placeholders.
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) #enable native 2x faster inference

messages = [
    {"role": "user", "content": message},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenizer = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = False, # Changed use_cache to False
                         temperature = 1.5, min_p = 0.1)
response = tokenizer.batch_decode(outputs)


In [ ]:
print(response[0])